# CUDA-Q Internal Representations

This notebook inspects the internal and external forms generated from one CUDA-Q kernel.

CUDA-Q maps quantum kernels to the Quake MLIR dialect. The same kernel can then be lowered to QIR or exported as OpenQASM 2 with `cudaq.translate()`. This notebook is not a full compiler tutorial; it is a quick way to see how the Bell kernel appears at each level.

References:

- [Working with the CUDA-Q IR](https://nvidia.github.io/cuda-quantum/latest/using/extending/cudaq_ir.html)
- [Quake Dialect](https://nvidia.github.io/cuda-quantum/latest/specification/quake-dialect.html)
- [cudaq.translate API](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.translate)

## MLIR

In [1]:
import cudaq


@cudaq.kernel
def kernel():
    q = cudaq.qvector(2)
    h(q[0])
    cx(q[0], q[1])


# Look at the MLIR
print(kernel)

module attributes {cc.python_uniqued = "kernel..0x110c93410", llvm.data_layout = "e-m:o-i64:64-i128:128-n32:64-S128", quake.mangled_name_map = {__nvqpp__mlirgen__kernel..0x110c93410 = "__nvqpp__mlirgen__kernel..0x110c93410_PyKernelEntryPointRewrite"}} {
  func.func @__nvqpp__mlirgen__kernel..0x110c93410() attributes {"cudaq-entrypoint", "cudaq-kernel"} {
    %0 = quake.alloca !quake.veq<2>
    %1 = quake.extract_ref %0[0] : (!quake.veq<2>) -> !quake.ref
    quake.h %1 : (!quake.ref) -> ()
    %2 = quake.extract_ref %0[0] : (!quake.veq<2>) -> !quake.ref
    %3 = quake.extract_ref %0[1] : (!quake.veq<2>) -> !quake.ref
    quake.x [%2] %3 : (!quake.ref, !quake.ref) -> ()
    quake.dealloc %0 : !quake.veq<2>
    return
  }
  func.func private @hybridLaunchKernel(!cc.ptr<i8>, !cc.ptr<i8>, !cc.ptr<i8>, i64, i64, !cc.ptr<i8>) -> !cc.struct<{!cc.ptr<i8>, i64}>
  func.func private @cudaqRegisterArgsCreator(!cc.ptr<i8>, !cc.ptr<i8>)
  llvm.func @cudaqRegisterLambdaName(!llvm.ptr<i8>, !llvm.ptr<i

The important lines in the MLIR output are:

- `quake.alloca !quake.veq<2>` allocates a two-qubit vector.
- `quake.extract_ref` extracts a reference to one qubit in that vector.
- `quake.h` applies the Hadamard gate.
- `quake.x [%control] target` applies a controlled-X operation, equivalent to `cx`.
- `quake.dealloc` releases the qubit vector.

## Translated Formats

`cudaq.translate()` returns a string representation of the kernel. The official Python API lists `qir`, `qir-full`, `qir-base`, `qir-adaptive`, and `openqasm2` as available format names.

- `qir` and its variants are LLVM/QIR forms used closer to the execution toolchain.
- `openqasm2` is a compact circuit exchange format that is easier to read for small gate-level circuits.
- OpenQASM 2 translation has limitations for kernels with arguments, so this notebook uses a no-argument kernel.

## QIR

In [2]:
# Look at the QIR
print(cudaq.translate(kernel, format="qir"))

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"
target datalayout = "e-m:o-i64:64-i128:128-n32:64-S128"

%Array = type opaque
%Qubit = type opaque

declare i8* @malloc(i64)

declare void @free(i8*)

define void @__nvqpp__mlirgen__kernel..0x110c93410() {
  %1 = call %Array* @__quantum__rt__qubit_allocate_array(i64 2)
  %2 = call %Qubit** @__quantum__rt__array_get_element_ptr_1d(%Array* %1, i64 0)
  %3 = load %Qubit*, %Qubit** %2, align 8
  call void @__quantum__qis__h(%Qubit* %3)
  %4 = call %Qubit** @__quantum__rt__array_get_element_ptr_1d(%Array* %1, i64 1)
  %5 = load %Qubit*, %Qubit** %4, align 8
  %6 = bitcast %Qubit* %3 to i8*
  %7 = bitcast %Qubit* %5 to i8*
  call void (i64, i64, i64, i64, i8*, ...) @generalizedInvokeWithRotationsControlsTargets(i64 0, i64 0, i64 1, i64 1, i8* bitcast (void (%Array*, %Qubit*)* @__quantum__qis__x__ctl to i8*), i8* %6, i8* %7)
  call void @__quantum__rt__qubit_release_array(%Array* %1)
  ret void
}

declare %Array* @__quantu

## OPENQASM 2

In [3]:
print(cudaq.translate(kernel, format="openqasm2"))

// Code generated by NVIDIA's nvq++ compiler
OPENQASM 2.0;

include "qelib1.inc";

qreg var0[2];
h var0[0];
cx var0[0], var0[1];



In [4]:
### Version information
print(cudaq.__version__)

CUDA-Q Version 0.14.2 (https://github.com/NVIDIA/cuda-quantum 91ab3092e76dab8887d1fdf0c99a2478ca90581c)
